Download data from [UK-based online retail](https://archive.ics.uci.edu/dataset/502/online+retail+ii).

future is 369 days

# Cleaning

## Importing Libraries and data

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#from tqdm import tqdm
#import re
import os
#from tqdm.notebook import tqdm
from tqdm.auto import tqdm
from collections import defaultdict

#from sentence_transformers import SentenceTransformer
#from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
#from sklearn.metrics import silhouette_score

In [2]:
print('Current working directory is', os.getcwd())

# Loading calibration data
df = pd.read_csv("data/calibration_raw.csv")

Current working directory is /Users/taniazamansarna/projects/customer-clv


In [3]:
df.head(4)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom


### Data type handling

In [4]:
df['Invoice'] = df['Invoice'].astype(str).str.upper().str.strip()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = df.rename(columns={"Customer ID": "Customer_ID"})
df["Customer_ID"] = df["Customer_ID"].astype("Int64")
df['StockCode'] = df['StockCode'].astype(str).str.upper().str.strip()
df['Description'] = df['Description'].astype(str).str.upper().str.strip()
df['Country'] = df['Country'].astype(str).str.upper().str.strip()

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517776 entries, 0 to 517775
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      517776 non-null  str           
 1   StockCode    517776 non-null  str           
 2   Description  514871 non-null  str           
 3   Quantity     517776 non-null  int64         
 4   InvoiceDate  517776 non-null  datetime64[us]
 5   Price        517776 non-null  float64       
 6   Customer_ID  412989 non-null  Int64         
 7   Country      517776 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 32.1 MB


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517776 entries, 0 to 517775
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      517776 non-null  str           
 1   StockCode    517776 non-null  str           
 2   Description  514871 non-null  str           
 3   Quantity     517776 non-null  int64         
 4   InvoiceDate  517776 non-null  datetime64[us]
 5   Price        517776 non-null  float64       
 6   Customer_ID  412989 non-null  Int64         
 7   Country      517776 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 32.1 MB


## Identifying and handling cancelled/ returned order

This step assigns a clear transaction status to each line item by matching negative-quantity rows against earlier positive-quantity rows from the same customer, product, quantity, and price.

The idea is simple:

- A **positive** quantity usually means a normal purchase.
- A **negative** quantity usually means a return, cancellation, or reversal.
- If a negative row can be matched to a previous positive row with the same customer, stock code, absolute quantity, and price, then the two rows belong together.
- The positive row is marked as **`cancelled`**.
- The negative row is marked as **`return: purchase found`**.
- If no matching purchase exists, the negative row is marked as **`return: purchase not found`**.
- Any remaining positive rows that were never matched to a return are marked as **`completed`**.

This approach is much faster than filtering the whole dataframe for every negative row. Instead, the data is processed once in chronological order, and a hash-based lookup structure stores unmatched positive rows. That makes matching efficient even for large datasets.

The `status` column is a new descriptive field that summarizes the outcome of the matching logic for each row. It is useful later for cleaning, analysis, CLV modeling, and churn/retention work because it separates normal purchases from returns and cancellations.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517776 entries, 0 to 517775
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      517776 non-null  str           
 1   StockCode    517776 non-null  str           
 2   Description  514871 non-null  str           
 3   Quantity     517776 non-null  int64         
 4   InvoiceDate  517776 non-null  datetime64[us]
 5   Price        517776 non-null  float64       
 6   Customer_ID  412989 non-null  Int64         
 7   Country      517776 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 32.1 MB


In [7]:
# Both negative price and negative quantity mean cancellation of orders; 
# I standardize cancellation of order by considering Quantity -ve and price positive
df['Quantity'] = df['Quantity'] * np.sign(df['Price'])
df['Price'] = np.abs(df['Price'])

In [8]:
# Sort by time
df = df.sort_values("InvoiceDate").reset_index(drop=True)

# Status column
df["status"] = pd.NA

# ---------------------------------------------------
# Matching structure:
# key = (Customer_ID, StockCode, abs(Quantity), Price)
# value = list of unmatched positive row positions
# ---------------------------------------------------
open_sales = defaultdict(list)

# Use tqdm for progress tracking
for i, row in tqdm(
    enumerate(df.itertuples(index=False)),
    total=len(df),
    desc="Matching returns"
):
    key = (
        row.Customer_ID,
        row.StockCode,
        abs(row.Quantity),
        row.Price
    )
    if row.Quantity > 0:
        open_sales[key].append(i)

    elif row.Quantity < 0:
        if open_sales[key]:
            match_i = open_sales[key].pop()
            df.at[match_i, "status"] = "cancelled"
            df.at[i, "status"] = "return: purchase found"
        else:
            df.at[i, "status"] = "return: purchase not found"

# Any remaining positive rows are completed purchases
df.loc[(df["Quantity"] > 0) & (df["status"].isna()), "status"] = "completed"

Matching returns:   0%|          | 0/517776 [00:00<?, ?it/s]

In [9]:
df['status'] = df['status'].astype(str)
df.info()

## Keeping only completed orders
df = df[df['status']=='completed']

print(f'Size of the calibration data after adjusting returned order is: {len(df)}')

<class 'pandas.DataFrame'>
RangeIndex: 517776 entries, 0 to 517775
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      517776 non-null  str           
 1   StockCode    517776 non-null  str           
 2   Description  514871 non-null  str           
 3   Quantity     517776 non-null  float64       
 4   InvoiceDate  517776 non-null  datetime64[us]
 5   Price        517776 non-null  float64       
 6   Customer_ID  412989 non-null  Int64         
 7   Country      517776 non-null  str           
 8   status       514177 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(2), str(5)
memory usage: 36.0 MB
Size of the calibration data after adjusting returned order is: 500948


In [10]:
# droping status column
df.drop(columns = 'status', inplace=True)

In [11]:
# Deleting lines that put as a test, or as fee	
df = df[~(df["Description"]=="CHECKED")]
df = df[~(df["Description"]=="CHECK")]
df = df[~(df["Description"]=="CHECK?")]
df = df[~(df["Description"]=="STOCK CHECK")]
df = df[~(df["Description"]=="CARRIAGE")]

df = df[~df["Description"].str.contains("damaged", case=False, na=False)]
df = df[~df["Description"].str.contains("BANK CHARGES", case=False, na=False)]
df = df[~df["Description"].str.contains("DISCOUNT", case=False, na=False)]
df = df[~df["Description"].str.contains("POSTAGE", case=False, na=False)]
df = df[~df["Description"].str.contains("MANUAL", case=False, na=False)]
df = df[~df["Description"].str.contains("ADJUST", case=False, na=False)]
df = df[~df["Description"].str.contains("test", case=False, na=False)]
df = df[~df["Description"].str.contains("lost", case=False, na=False)]
df = df[~df["Description"].str.contains("samples", case=False, na=False)]

# 5. Remove administrative/service stock codes
non_product_codes = ['DOT', 'POST', 'C2', 'M', 'BANK CHARGES', 'PADS', 'CRUK']
df = df[~df['StockCode'].isin(non_product_codes)]

df = df[~df["StockCode"].str.contains("test", case=False, na=False)]
df = df[~df["StockCode"].str.contains("fee", case=False, na=False)]

## BASIC CLEANING
Before moving to modeling, the dataset needs a few consistency checks so each transaction line can be trusted as a valid customer purchase record.

First, the relationship between `Customer_ID` and `Country` is checked. In a clean transactional dataset, the same customer should usually belong to one country. If a customer appears under multiple countries, the code identifies those cases and resolves them by assigning the most representative country for that customer. The chosen country is the one linked to the highest number of invoices, and if there is a tie, the one with the highest total quantity.

After that, the code checks whether each invoice belongs to only one customer. A single invoice should normally map to a single `Customer_ID`. If one invoice is linked to multiple customer IDs, those cases are flagged for review.

Finally, the `Description` column is standardized. The same `StockCode` should generally refer to the same product name, but inconsistent descriptions can appear in raw retail data. To fix this, the most common description for each `StockCode` is kept and assigned back to the dataframe.

These cleaning steps make the dataset more consistent and help ensure that later analyses are based on stable customer, product, and transaction definitions.

### Handling mismatches between customer and country columns

In [12]:
###  Checking if customer and country mismatch exists

# Creating new dataframe by dropping rows with blank customer_ID for checking customer country mismatch check
rows_with_customer = df[df['Customer_ID'].notna()].copy()

# Count unique Country per Customer_ID
customer_country_check = (
    rows_with_customer
    .groupby('Customer_ID')
    .agg(
        n_countries=('Country', lambda x: x.nunique(dropna=False))
    )
    .reset_index()
)

# Customers linked to more than one country
country_mismatch = customer_country_check[
    customer_country_check['n_countries'] > 1
]

# Save all original rows belonging to those multiple country customers
customer_country_mismatches = (
    rows_with_customer[
        rows_with_customer['Customer_ID'].isin(country_mismatch['Customer_ID'])
    ]
    .sort_values(['Customer_ID', 'Country', 'InvoiceDate'])
)

if country_mismatch.shape[0] == 0:
    print('No mismatch between Customer_ID and Country')
else:
    print(f'❌ Country column of {country_mismatch.shape[0]} customers are not unique between different lines. \n \n ID\'s of the customres are:\n')
    print(customer_country_mismatches[
    ['Customer_ID', 'Country']
].drop_duplicates().sort_values(['Customer_ID', 'Country']).head(50))

❌ Country column of 4 customers are not unique between different lines. 
 
 ID's of the customres are:

        Customer_ID      Country
12492         12417      BELGIUM
365828        12417        SPAIN
39830         12422    AUSTRALIA
205288        12422  SWITZERLAND
467038        12423      BELGIUM
321987        12423      DENMARK
110889        12431    AUSTRALIA
57311         12431      BELGIUM


In [13]:
### Resolving mismatch between country and customer

# Invoice count and total quantity by Customer_ID
customer_country_stats = (
    rows_with_customer
    .groupby(['Customer_ID', 'Country'])
    .agg(
        n_invoices=('Invoice', 'nunique'),
        total_quantity=('Quantity', 'sum')
    )
    .reset_index()
)

# Pick mode country:
# 1. highest number of invoices
# 2. if tied, highest total quantity
mode_country_per_customer = (
    customer_country_stats
    .sort_values(
        ['Customer_ID', 'n_invoices', 'total_quantity'],
        ascending=[True, False, False]
    )
    .drop_duplicates('Customer_ID', keep='first')
    [['Customer_ID', 'Country']]
    .rename(columns={'Country': 'mode_country'})
)

# Add mode country to original df
df = df.merge(
    mode_country_per_customer,
    on='Customer_ID',
    how='left'
)

# Replace Country only for rows with given Customer_ID
df.loc[df['Customer_ID'].notna(), 'Country'] = df.loc[
    df['Customer_ID'].notna(),
    'mode_country'
]

# Drop helper column
df = df.drop(columns='mode_country')

print('Mismatches resolved between Customer_ID and Country, by updating country column. ✅')

Mismatches resolved between Customer_ID and Country, by updating country column. ✅


### Handling duplicate rows

In [14]:
### Checking whether rows corresponds to a same invoice referes to a same Customer_ID 

# Count unique Customer_ID  per Invoice
# dropna=False means missing Customer_ID is counted as its own value
invoice_check = (
    df.groupby('Invoice')
      .agg(
          n_customers=('Customer_ID', lambda x: x.nunique(dropna=False))
      )
      .reset_index()
)

# Invoices where same invoice has different customer 
bad_invoices = invoice_check[(invoice_check['n_customers'] > 1)]

# Save all original rows belonging to those problematic invoices
invoice_exceptions = (
    df[df['Invoice'].isin(bad_invoices['Invoice'])]
    .sort_values(['Invoice', 'Customer_ID'])
)


if bad_invoices.shape[0] == 0:
    print('No mismatch found between invoice and Customer_ID.  ✅')
else:
    print(f'{bad_invoices.shape[0]} mismatches observed between invoice and Customer_ID. Some mismatches as examples are:')
    print(invoice_exceptions[['Invoice', 'Customer_ID', 'Country']].drop_duplicates().head(5))

No mismatch found between invoice and Customer_ID.  ✅


In [15]:
# Number of duplicate rows (considering all columns)
n_duplicates = df.duplicated().sum()

if n_duplicates == 0:
    print("✅ No duplicate rows found.")
else:
    print(f"❌ Found {n_duplicates:,} duplicate rows.")

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)

    print(f"Deleted {n_duplicates:,} duplicate rows.")
    print(f"Remaining rows: {len(df):,}")

❌ Found 13,805 duplicate rows.
Deleted 13,805 duplicate rows.
Remaining rows: 484,985


```text 
(Invoice, StockCode) combination supposed be unique across the data. We found 5,726 duplicate (Invoice, StockCode) pairs.

This behavior is expected in transaction-level retail data because an invoice may contain multiple line items for the same product due to separate entries, adjustments, or ERP order-line recording. Since customer value analysis is performed at the invoice level, these records will be retained and aggregated in the next steps.
```

In [16]:
# Total rows
n_rows = len(df)

# Number of unique (Invoice, StockCode) pairs
n_unique = df[['Invoice', 'StockCode']].drop_duplicates().shape[0]

print(f"Total rows: {n_rows:,}")
print(f"Unique (Invoice, StockCode) pairs: {n_unique:,}")

if n_rows == n_unique:
    print("✅ Now, every (Invoice, StockCode)  is unique.")
else:
    print(f"❌ Found {n_rows - n_unique:,} duplicate (Invoice, StockCode) pairs.")

Total rows: 484,985
Unique (Invoice, StockCode) pairs: 478,907
❌ Found 6,078 duplicate (Invoice, StockCode) pairs.


In [17]:
df["line_amount"] = df["Quantity"] * df["Price"]

df = (
    df.groupby(["Invoice", "StockCode"], as_index=False)
      .agg(
          Quantity=("Quantity", "sum"),
          line_amount=("line_amount", "sum"),
          Description=("Description", "first"),
          InvoiceDate=("InvoiceDate", "first"),
          **{"Customer_ID": ("Customer_ID", "first")},
          Country=("Country", "first")
      )
)

df["Price"] = df["line_amount"] / df["Quantity"]

# Total rows
n_rows = len(df)

# Number of unique (Invoice, StockCode) pairs
n_unique = df[['Invoice', 'StockCode']].drop_duplicates().shape[0]

if n_rows == n_unique:
    print("After cleaning, (Invoice, StockCode)  is unique for every lines. ✅")
    print(f"Total lines after handling duplicates is: {n_unique:,}")
else:
    print(f"❌ Found {n_rows - n_unique:,} duplicate (Invoice, StockCode) pairs from total rows: {n_rows:,}.")

After cleaning, (Invoice, StockCode)  is unique for every lines. ✅
Total lines after handling duplicates is: 478,907


## Creating and Saving data for product recommendation analysis

In [18]:
collapsed = (
    df.groupby(['StockCode', 'Description'])
      .size()
      .rename('count')
      .reset_index()
)

collapsed['rank'] = (
    collapsed.groupby('StockCode')['count']
             .rank(method='first', ascending=False)
)

collapsed = collapsed[collapsed['rank'] == 1]

df = df.merge(collapsed, on='StockCode', how='left', suffixes=('', '_top'))
df['Description'] = df['Description_top']
df = df.drop(columns=['Description_top', 'rank', 'count'])

In [19]:
tmp = df.groupby("StockCode").agg(minprice=("Price","min"), maxprice=("Price","max"), avgprice=("Price","mean"), std_of_price=("Price","std"))
desc = df.groupby(["StockCode","Description"]).size().reset_index(name="n").sort_values(["StockCode","n"], ascending=[True,False]).drop_duplicates("StockCode").set_index("StockCode")[["Description"]]
df_product = desc.join(tmp).reset_index()
df_product["std_of_price"] = df_product["std_of_price"].fillna(0)
df_product.head()

#df.drop(columns='Description', inplace=True)

df.to_csv('data/calib_all_transaction.csv', index=False)
df_product.to_csv('data/prod_catalog_calib.csv', index=False)

## Creating and Saving data for CLV analysis

### Calibration data

In [20]:
# Number of rows before removing rows with missing Customer_ID
print(f"Rows before removing rows with missing Customer_ID : {len(df)}")

# Remove rows with missing Customer_ID
df = df[df["Customer_ID"].notna()].copy()

print(f"Rows remaining                                     : {len(df)}")

df.info()

Rows before removing rows with missing Customer_ID : 478907
Rows remaining                                     : 381765
<class 'pandas.DataFrame'>
Index: 381765 entries, 0 to 478906
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      381765 non-null  str           
 1   StockCode    381765 non-null  str           
 2   Quantity     381765 non-null  float64       
 3   line_amount  381765 non-null  float64       
 4   Description  381765 non-null  str           
 5   InvoiceDate  381765 non-null  datetime64[us]
 6   Customer_ID  381765 non-null  Int64         
 7   Country      381765 non-null  str           
 8   Price        381765 non-null  float64       
dtypes: Int64(1), datetime64[us](1), float64(3), str(4)
memory usage: 29.5 MB


In [21]:
## creating invoice table: Group by customer and invoice, then sum the costs and quantities
df_invoice = df.groupby(['Invoice', 'Customer_ID','InvoiceDate', 'Country']).agg(
    total_amount=('line_amount', 'sum'),
    number_of_items=('Quantity', 'sum')
).reset_index()

# make a copy
df_order = df_invoice.copy()

# extract date only from date-time
df_order["order_date"] = df_order["InvoiceDate"].dt.date

# group transation lines with same customer + same day into one order
df_order = (
    df_order
    .groupby(["Customer_ID", "order_date"], as_index=False)
    .agg(
        total_amount=("total_amount", "sum"),
        number_of_items=("number_of_items", "sum"),
        Country=("Country", "first"),
        InvoiceDate=('InvoiceDate', 'first')
    )
)

# create order_id as string: Customer_ID + date
df_order["order_id"] = (
    df_order["Customer_ID"].astype(str) + "_" +
    pd.to_datetime(df_order["order_date"]).dt.strftime("%Y-%m-%d")
)

df_order.drop(columns='InvoiceDate', inplace=True)

df_order['order_date'] = pd.to_datetime(df_order['order_date'])

print('df_order')
df_order.info()

df_order.to_csv('data/calib_order.csv', index=False)

df_order
<class 'pandas.DataFrame'>
RangeIndex: 16368 entries, 0 to 16367
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype        
---  ------           --------------  -----        
 0   Customer_ID      16368 non-null  Int64        
 1   order_date       16368 non-null  datetime64[s]
 2   total_amount     16368 non-null  float64      
 3   number_of_items  16368 non-null  float64      
 4   Country          16368 non-null  str          
 5   order_id         16368 non-null  str          
dtypes: Int64(1), datetime64[s](1), float64(2), str(2)
memory usage: 783.4 KB


### Holdout data

In [22]:
df_holdout = pd.read_csv('data/holdout_transaction.csv')
df_holdout['InvoiceDate'] = pd.to_datetime(df_holdout['InvoiceDate'])

df_holdout.info()

<class 'pandas.DataFrame'>
RangeIndex: 512647 entries, 0 to 512646
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      512647 non-null  int64         
 1   StockCode    512647 non-null  str           
 2   Description  512647 non-null  str           
 3   Quantity     512647 non-null  float64       
 4   InvoiceDate  512647 non-null  datetime64[us]
 5   Price        512647 non-null  float64       
 6   Customer_ID  383580 non-null  float64       
 7   Country      512647 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 31.3 MB


In [23]:
# Number of rows before removing rows with missing Customer_ID
print(f"Rows before removing rows with missing Customer_ID : {len(df_holdout)}")

# Remove rows with missing Customer_ID
df_holdout = df_holdout[df_holdout["Customer_ID"].notna()].copy()

print(f"Rows remaining                                     : {len(df_holdout)}")

df_holdout.info()

Rows before removing rows with missing Customer_ID : 512647
Rows remaining                                     : 383580
<class 'pandas.DataFrame'>
Index: 383580 entries, 0 to 512646
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      383580 non-null  int64         
 1   StockCode    383580 non-null  str           
 2   Description  383580 non-null  str           
 3   Quantity     383580 non-null  float64       
 4   InvoiceDate  383580 non-null  datetime64[us]
 5   Price        383580 non-null  float64       
 6   Customer_ID  383580 non-null  float64       
 7   Country      383580 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 26.3 MB


In [24]:
# Line amount
df_holdout["line_amount"] = df_holdout["Quantity"] * df_holdout["Price"]


## creating invoice table: Group by customer and invoice, then sum the costs and quantities
df_holdout_invoice = df_holdout.groupby(['Invoice', 'Customer_ID','InvoiceDate', 'Country']).agg(
    total_amount=('line_amount', 'sum'),
    number_of_items=('Quantity', 'sum')
).reset_index()

# make a copy
df_holdout_order = df_holdout_invoice.copy()

# extract date only from date-time
df_holdout_order["order_date"] = df_holdout_order["InvoiceDate"].dt.date

# group transation lines with same customer + same day into one order
df_holdout_order = (
    df_holdout_order
    .groupby(["Customer_ID", "order_date"], as_index=False)
    .agg(
        total_amount=("total_amount", "sum"),
        number_of_items=("number_of_items", "sum"),
        Country=("Country", "first"),
        InvoiceDate=('InvoiceDate', 'first')
    )
)

# create order_id as string: Customer_ID + date
df_holdout_order["order_id"] = (
    df_holdout_order["Customer_ID"].astype(str) + "_" +
    pd.to_datetime(df_holdout_order["order_date"]).dt.strftime("%Y-%m-%d")
)

df_holdout_order.drop(columns='InvoiceDate', inplace=True)

df_holdout_order['order_date'] = pd.to_datetime(df_holdout_order['order_date'])

print('df_holdout_order')
df_holdout_order.info()

df_holdout_order.to_csv('data/holdout_order.csv', index=False)

df_holdout_order
<class 'pandas.DataFrame'>
RangeIndex: 16363 entries, 0 to 16362
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype        
---  ------           --------------  -----        
 0   Customer_ID      16363 non-null  float64      
 1   order_date       16363 non-null  datetime64[s]
 2   total_amount     16363 non-null  float64      
 3   number_of_items  16363 non-null  float64      
 4   Country          16363 non-null  str          
 5   order_id         16363 non-null  str          
dtypes: datetime64[s](1), float64(3), str(2)
memory usage: 767.1 KB
